# Surreal ORM Lite - Getting Started

> **Note:** These examples target **surreal-orm-lite v0.7.0+**, which uses the official **SurrealDB Python SDK 2.x** (`surrealdb[pydantic]>=2.0.0,<3.0.0`) and supports the **SurrealDB 3.x** protocol. It is tested against SurrealDB **v2.6.5** and **v3.1.3**. See the [CHANGELOG](../CHANGELOG.md) for version-specific details.
>
> Since v0.7.0, a record loaded from the database exposes its `id` as a native `surrealdb.RecordID` **when the model field is typed to accept it** (e.g. `id: str | RecordID | None`). Use `model.get_raw_id()` to get the bare identifier string. Models that declare `id: str` (as below) keep receiving a plain string for convenience.

This notebook demonstrates how to use `surreal-orm-lite`, a lightweight Django-style ORM for SurrealDB using the official Python SDK.

## Prerequisites

1. Install the package:
```bash
pip install surreal-orm-lite
```

2. Start a SurrealDB instance (v2.6.x or v3.x):
```bash
docker run -d -p 8000:8000 surrealdb/surrealdb:v3.1.3 start --user root --pass root
```

## 1. Setup Connection

In [1]:
from surreal_orm_lite import SurrealDBConnectionManager

# Configure the connection (does not connect yet)
SurrealDBConnectionManager.set_connection(
    url="http://localhost:8000",
    user="root",
    password="root",
    namespace="test",
    database="test"
)

print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Define a Model

Models inherit from `BaseSurrealModel` and use Pydantic for validation.

In [2]:
from pydantic import Field
from surreal_orm_lite import BaseSurrealModel


class User(BaseSurrealModel):
    """User model - table name will be 'User' (class name)"""
    id: str | None = None  # Optional: auto-generated if not provided
    name: str = Field(..., max_length=100)
    email: str = Field(..., max_length=255)
    age: int = Field(..., ge=0, le=150)
    active: bool = Field(default=True)

## 3. Create Records (INSERT)

In [3]:
# Create a user with a specific ID
alice = User(id="1", name="Alice", email="alice@example.com", age=30)
await alice.save()
print(f"Created: {alice}")

# Create more users
bob = User(id="2", name="Bob", email="bob@example.com", age=25)
await bob.save()

charlie = User(id="3", name="Charlie", email="charlie@example.com", age=35, active=False)
await charlie.save()

print("Users created!")

Created: id='1' name='Alice' email='alice@example.com' age=30 active=True
Users created!


## 4. Read Records (SELECT)

In [4]:
# Get all users
all_users = await User.objects().all()
print(f"Total users: {len(all_users)}")
for user in all_users:
    print(f"  - {user.name} ({user.email})")

Total users: 3
  - Alice (alice@example.com)
  - Bob (bob@example.com)
  - Charlie (charlie@example.com)


In [5]:
# Get a specific user by ID
user = await User.objects().get("1")
print(f"Found: {user.name}, Age: {user.age}")

Found: Alice, Age: 30


In [6]:
# Get first matching result
first_user = await User.objects().filter(active=True).first()
print(f"First active user: {first_user.name}")

First active user: Alice


## 5. Filter Queries

Supported lookups:
- `exact` (default): `name="Alice"`
- `gt`: greater than
- `gte`: greater than or equal
- `lt`: less than
- `lte`: less than or equal
- `in`: within a list
- `contains`: string contains

In [7]:
# Filter by exact value
alice = await User.objects().filter(name="Alice").first()
print(f"Found Alice: {alice.email}")

Found Alice: alice@example.com


In [8]:
# Filter with comparison operators
young_users = await User.objects().filter(age__lt=30).exec()
print(f"Users under 30: {[u.name for u in young_users]}")

adult_users = await User.objects().filter(age__gte=25).exec()
print(f"Users 25 or older: {[u.name for u in adult_users]}")

Users under 30: ['Bob']
Users 25 or older: ['Alice', 'Bob', 'Charlie']


In [9]:
# Filter with IN operator
specific_users = await User.objects().filter(name__in=["Alice", "Bob"]).exec()
print(f"Alice or Bob: {[u.name for u in specific_users]}")

Alice or Bob: ['Alice', 'Bob']


In [10]:
# Combine multiple filters (AND)
filtered = await User.objects().filter(age__gte=25, active=True).exec()
print(f"Active users 25+: {[u.name for u in filtered]}")

Active users 25+: ['Alice', 'Bob']


## 6. Ordering, Limit & Offset

In [11]:
from surreal_orm_lite import OrderBy

# Order by name ascending
sorted_users = await User.objects().order_by("name").exec()
print(f"Sorted A-Z: {[u.name for u in sorted_users]}")

# Order by age descending
oldest_first = await User.objects().order_by("age", OrderBy.DESC).exec()
print(f"Oldest first: {[f'{u.name}({u.age})' for u in oldest_first]}")

Sorted A-Z: ['Alice', 'Bob', 'Charlie']
Oldest first: ['Charlie(35)', 'Alice(30)', 'Bob(25)']


In [12]:
# Pagination with limit and offset
page1 = await User.objects().order_by("name").limit(2).exec()
print(f"Page 1: {[u.name for u in page1]}")

page2 = await User.objects().order_by("name").limit(2).offset(2).exec()
print(f"Page 2: {[u.name for u in page2]}")

Page 1: ['Alice', 'Bob']
Page 2: ['Charlie']


## 7. Select Specific Fields

In [13]:
# Select only specific fields (returns dict, not model)
names_only = await User.objects().select("name", "email").exec()
print(f"Names and emails: {names_only}")

Names and emails: [{'email': 'alice@example.com', 'name': 'Alice'}, {'email': 'bob@example.com', 'name': 'Bob'}, {'email': 'charlie@example.com', 'name': 'Charlie'}]


## 8. Update Records

In [14]:
# Full update - replaces all fields
user = await User.objects().get("1")
user.age = 31
await user.update()

# Verify
updated = await User.objects().get("1")
print(f"Alice's new age: {updated.age}")

Alice's new age: 31


In [15]:
# Partial update with merge - only updates specified fields
user = await User.objects().get("2")
await user.merge(age=26, active=False)

# Verify
updated = await User.objects().get("2")
print(f"Bob: age={updated.age}, active={updated.active}")

Bob: age=26, active=False


## 9. Delete Records

In [16]:
# Delete a specific record
user = await User.objects().get("3")
await user.delete()

# Verify
remaining = await User.objects().all()
print(f"Remaining users: {[u.name for u in remaining]}")

Remaining users: ['Alice', 'Bob']


## 10. Custom Queries with Variables

In [17]:
# Use variables for parameterized queries
results = await User.objects().filter(age__gte="$min_age").variables(min_age=25).exec()
print(f"Users with age >= 25: {[u.name for u in results]}")

Users with age >= 25: ['Alice', 'Bob']


In [18]:
# Execute raw SurrealQL query
results = await User.objects().query(
    "SELECT * FROM User WHERE age > $age",
    {"age": 20}
)
print(f"Custom query results: {[u.name for u in results]}")

Custom query results: ['Alice', 'Bob']


## 11. Custom Primary Key

Use `SurrealConfigDict` to specify a custom primary key field.

In [19]:
from surreal_orm_lite import SurrealConfigDict


class Product(BaseSurrealModel):
    """Product model using SKU as primary key"""
    model_config = SurrealConfigDict(primary_key="sku")
    
    sku: str = Field(..., max_length=50)
    name: str = Field(..., max_length=200)
    price: float = Field(..., ge=0)


# Create product
laptop = Product(sku="LAPTOP-001", name="Gaming Laptop", price=1299.99)
await laptop.save()

# Fetch by SKU (use backticks for special characters)
product = await Product.objects().get("`LAPTOP-001`")
print(f"Product: {product.name} - ${product.price}")

# Cleanup
await Product.objects().delete_table()

Product: Gaming Laptop - $1299.99


True

## 12. Context Manager

In [20]:
# Use context manager for automatic connection handling
async with SurrealDBConnectionManager():
    users = await User.objects().all()
    print(f"Users in context: {len(users)}")

print(f"Connected after context: {SurrealDBConnectionManager.is_connected()}")

Users in context: 2
Connected after context: False


## 13. Aggregations *(v0.3.0+)*

Perform database-level calculations without loading all records into memory.

In [21]:
# Clean up existing data first, then re-create for aggregation demos
await User.objects().delete_table()

await User(id="1", name="Alice", email="alice@example.com", age=30, active=True).save()
await User(id="2", name="Bob", email="bob@example.com", age=25, active=True).save()
await User(id="3", name="Charlie", email="charlie@example.com", age=35, active=False).save()
await User(id="4", name="Diana", email="diana@example.com", age=28, active=True).save()

# Count all users
count = await User.objects().count()
print(f"Total users: {count}")

# Count with filter
active_count = await User.objects().filter(active=True).count()
print(f"Active users: {active_count}")

# Sum, Avg, Min, Max
total_age = await User.objects().sum("age")
avg_age = await User.objects().avg("age")
min_age = await User.objects().min("age")
max_age = await User.objects().max("age")
print(f"Age stats: sum={total_age}, avg={avg_age}, min={min_age}, max={max_age}")

# Check existence
has_inactive = await User.objects().filter(active=False).exists()
print(f"Has inactive users: {has_inactive}")

Total users: 4
Active users: 3


Age stats: sum=118, avg=29.5, min=25, max=35
Has inactive users: True


In [22]:
from surreal_orm_lite import Count, Sum, Avg

# GROUP BY with annotations
results = await User.objects().values("active").annotate(
    count=Count(),
    avg_age=Avg("age"),
    total_age=Sum("age"),
).exec()

for r in results:
    status = "Active" if r["active"] else "Inactive"
    print(f"{status}: count={r['count']}, avg_age={r['avg_age']}, total_age={r['total_age']}")

Inactive: count=1, avg_age=35.0, total_age=35
Active: count=3, avg_age=27.666666666666668, total_age=83


In [23]:
# raw_query() - Execute arbitrary SurrealQL with safe variable binding
results = await User.raw_query(
    "SELECT * FROM User WHERE age > $min_age AND active = $active",
    variables={"min_age": 25, "active": True}
)
print(f"Active users over 25: {[u.name for u in results if isinstance(u, User)]}")

# Aggregation raw query
stats = await User.raw_query("SELECT count() AS total FROM User GROUP ALL")
if stats and isinstance(stats[0], dict):
    print(f"Total via raw query: {stats[0]['total']}")

Active users over 25: ['Alice', 'Diana']


Total via raw query: 4


## 14. Model Signals *(v0.4.0+)*

Register async callbacks on model lifecycle events. Signals are fired automatically when `save()`, `update()`, `merge()`, or `delete()` are called.

### Pre/Post Signals

| Signal        | When                        | Extra kwargs    |
| ------------- | --------------------------- | --------------- |
| `pre_save`    | Before `save()`             |                 |
| `post_save`   | After `save()`              | `created`       |
| `pre_update`  | Before `update()`/`merge()` | `update_fields` |
| `post_update` | After `update()`/`merge()`  | `update_fields` |
| `pre_delete`  | Before `delete()`           |                 |
| `post_delete` | After `delete()`            |                 |

In [24]:
from surreal_orm_lite import pre_save, post_save, pre_delete, post_delete

# Register a pre_save handler: normalize email to lowercase before saving
@pre_save.connect(User)
async def normalize_email(sender, instance, **kwargs):
    original = instance.email
    object.__setattr__(instance, "email", instance.email.lower())
    if original != instance.email:
        print(f"  [pre_save] Normalized email: {original} -> {instance.email}")

# Register a post_save handler: log after successful save
@post_save.connect(User)
async def log_save(sender, instance, created, **kwargs):
    action = "Created" if created else "Saved"
    print(f"  [post_save] {action} user: {instance.name} ({instance.email})")

# Register a post_delete handler: log after deletion
@post_delete.connect(User)
async def log_delete(sender, instance, **kwargs):
    print(f"  [post_delete] Deleted user: {instance.name}")

# Clean table first
await User.objects().delete_table()

# Now save - signals fire automatically
print("Saving user with UPPER@CASE.COM email:")
user = User(id="s1", name="Eve", email="EVE@EXAMPLE.COM", age=22)
await user.save()

# Update triggers pre_update / post_update (not pre_save)
print("\nDeleting user:")
await user.delete()

# Cleanup handlers so they don't affect later cells
pre_save.clear(User)
post_save.clear(User)
post_delete.clear(User)

Saving user with UPPER@CASE.COM email:
  [pre_save] Normalized email: EVE@EXAMPLE.COM -> eve@example.com


  [post_save] Created user: Eve (eve@example.com)

Deleting user:
  [post_delete] Deleted user: Eve


### Around Signals

Around signals use async generators to wrap an operation. Code before `yield` runs before the operation, code after `yield` runs after. Useful for timing, logging, or wrapping in try/except.

In [25]:
import time
from surreal_orm_lite import around_save

@around_save.connect(User)
async def time_save(sender, instance, **kwargs):
    start = time.time()
    print(f"  [around_save] Starting save for {instance.name}...")
    yield  # <-- save() executes here
    duration = time.time() - start
    print(f"  [around_save] Save completed in {duration:.4f}s")

user = User(id="t1", name="Timed", email="timed@example.com", age=40)
await user.save()

# Cleanup
await User.objects().delete_table()
around_save.clear(User)

  [around_save] Starting save for Timed...


  [around_save] Save completed in 0.0028s


## 15. Cleanup

In [26]:
# Delete the entire table
await User.objects().delete_table()
print("User table deleted!")

# Close connection
await SurrealDBConnectionManager.close_connection()
print(f"Connected: {SurrealDBConnectionManager.is_connected()}")

User table deleted!
Connected: False


## Summary

| Operation | Method |
|-----------|--------|
| Create | `model.save()` |
| Read all | `Model.objects().all()` |
| Read one | `Model.objects().get(id)` |
| Read first | `Model.objects().filter(...).first()` |
| Filter | `Model.objects().filter(field=value)` |
| Update | `model.update()` |
| Partial update | `model.merge(field=value)` |
| Delete | `model.delete()` |
| Order | `Model.objects().order_by(field, OrderBy.DESC)` |
| Limit | `Model.objects().limit(n)` |
| Offset | `Model.objects().offset(n)` |
| Select fields | `Model.objects().select('f1', 'f2')` |
| Raw query | `Model.objects().query(sql, vars)` |
| Delete table | `Model.objects().delete_table()` |
| Count | `Model.objects().count()` |
| Sum | `Model.objects().sum("field")` |
| Average | `Model.objects().avg("field")` |
| Min / Max | `Model.objects().min("field")` / `.max("field")` |
| Exists | `Model.objects().filter(...).exists()` |
| GROUP BY | `Model.objects().values("field").annotate(count=Count()).exec()` |
| Raw SurrealQL | `Model.raw_query("SELECT ...", variables={...})` |
| Signal connect | `@pre_save.connect(Model)` |
| Signal clear | `pre_save.clear(Model)` |

For more advanced features (relations, transactions, etc.), see [SurrealDB-ORM](https://github.com/EulogySnowfall/SurrealDB-ORM/).